# Flower Image Classification: Milestone 1

This notebook builds a small PyTorch image-classification pipeline from scratch. It uses the labeled `train` and `valid` folders. The `test` folder contains unlabeled images, so it is checked and counted but never used for training or vaenter code herelidation.

In [3]:
from pathlib import Path
import zipfile

zip_path = Path(r"C:\Users\Asus\Downloads\flower_data.zip")
extract_path = Path("flower_data")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted to:", extract_path.resolve())

Dataset extracted to: C:\Users\Asus\Desktop\Machine-Learning-Project\flower_data


In [10]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import torch
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms


In [11]:
data_path = Path(r"C:\Users\Asus\Desktop\Machine-Learning-Project\flower_data\flower_data")
train_path = data_path / "train"
valid_path = data_path / "valid"
test_path = data_path / "test"
class_names_path = data_path / "cat_to_name.json"

image_size = 128
batch_size = 32
num_workers = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Training folder exists:", train_path.exists())
print("Validation folder exists:", valid_path.exists())
print("Test folder exists:", test_path.exists())
print("Class-name file exists:", class_names_path.exists())

if not train_path.exists() or not valid_path.exists():
    raise FileNotFoundError("Check that data_path points to the flower_data folder containing train and valid.")


Device: cuda
Training folder exists: True
Validation folder exists: True
Test folder exists: True
Class-name file exists: True


## 2. Read classes and image paths

Folder names are class IDs. The mapping is created from the training folders and reused for validation, so the same flower always has the same number in both splits. The helper accepts common image extensions without caring about upper/lowercase.

In [13]:

IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff", ".webp"]

def get_image_paths(folder):
    image_paths = []

    for path in folder.iterdir():
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            image_paths.append(path)

    return sorted(image_paths)


class_names = []

for folder in train_path.iterdir():
    if folder.is_dir():
        class_names.append(folder.name)

class_names.sort(key=lambda name: (0, int(name), "") if name.isdigit() else (1, 0, name))

class_to_idx = {}

for index, class_name in enumerate(class_names):
    class_to_idx[class_name] = index

idx_to_class = {}

for class_name, index in class_to_idx.items():
    idx_to_class[index] = class_name


valid_class_names = []

for folder in valid_path.iterdir():
    if folder.is_dir():
        valid_class_names.append(folder.name)

if set(class_names) != set(valid_class_names):
    raise ValueError("Train and validation must have the same class folders.")


class_to_name = {}

if class_names_path.exists():
    with open(class_names_path, "r", encoding="utf-8") as file:
        class_to_name = json.load(file)


train_samples = []
valid_samples = []

for class_name in class_names:
    class_number = class_to_idx[class_name]

    train_folder = train_path / class_name
    valid_folder = valid_path / class_name

    for image_path in get_image_paths(train_folder):
        train_samples.append((image_path, class_number))

    for image_path in get_image_paths(valid_folder):
        valid_samples.append((image_path, class_number))


if len(train_samples) == 0 or len(valid_samples) == 0:
    raise ValueError("No images found. Check the dataset folders.")

print("Number of classes:", len(class_names))
print("Training images:", len(train_samples))
print("Validation images:", len(valid_samples))
print("Class mapping:", class_to_idx)

if test_path.exists():
    test_images = get_image_paths(test_path)
    print("Test images:", len(test_images))


Number of classes: 102
Training images: 6552
Validation images: 818
Class mapping: {'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, '10': 9, '11': 10, '12': 11, '13': 12, '14': 13, '15': 14, '16': 15, '17': 16, '18': 17, '19': 18, '20': 19, '21': 20, '22': 21, '23': 22, '24': 23, '25': 24, '26': 25, '27': 26, '28': 27, '29': 28, '30': 29, '31': 30, '32': 31, '33': 32, '34': 33, '35': 34, '36': 35, '37': 36, '38': 37, '39': 38, '40': 39, '41': 40, '42': 41, '43': 42, '44': 43, '45': 44, '46': 45, '47': 46, '48': 47, '49': 48, '50': 49, '51': 50, '52': 51, '53': 52, '54': 53, '55': 54, '56': 55, '57': 56, '58': 57, '59': 58, '60': 59, '61': 60, '62': 61, '63': 62, '64': 63, '65': 64, '66': 65, '67': 66, '68': 67, '69': 68, '70': 69, '71': 70, '72': 71, '73': 72, '74': 73, '75': 74, '76': 75, '77': 76, '78': 77, '79': 78, '80': 79, '81': 80, '82': 81, '83': 82, '84': 83, '85': 84, '86': 85, '87': 86, '88': 87, '89': 88, '90': 89, '91': 90, '92': 91, '93': 92, '94':


## 3. Image transformations and custom Dataset

Each image is converted to RGB, resized to one shared size, then converted to a tensor. A single image has shape `(channels, height, width)`, which is `(3, 128, 128)` here. The label is an integer class index.

In [14]:
image_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
])

class FlowerDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]
        with Image.open(image_path) as image_file:
            image = image_file.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label

train_dataset = FlowerDataset(train_samples, image_transform)
valid_dataset = FlowerDataset(valid_samples, image_transform)
print("One item image shape:", train_dataset[0][0].shape)
print("One item label:", train_dataset[0][1])


One item image shape: torch.Size([3, 128, 128])
One item label: 0
